# Chiseling 101

`CC-BY 2026 Brooksbank, Kassabov, Wilson`

Dleto (Chisel) for Python detects and locates sparsity patterns in tensor data.  This notebook demonstrates the basic functionality of Dleto, how to select different chisels, and what to expect from the outputs.  

1. [Tucker Decompositions](#1-tucker-decompositions)
2. [Block Decompositions](#2-block-diagonalization)
3. [Stratification](#3-stratification)


## 1. Tucker Decompositions

We first use Dleto to find familiar tensor structures known as Tucker Decompositions. You can think of these as tensor generalizations of matrix nullspaces. 

**Performance.** Tucker decompositions have been explored since the 1800's, and there are highly optimized strategies to discover them. This tutorial uses a familiar problem to explore the Dleto's functionality, but the performance of our demonstration is far from optimal on this particular application. Due to its generality, Dleto chiseling operates with a complexity slightly greater than more direct Tucker Decomposition strategies. Indeed Dleto itself leverages such strategies through its function `nondeg`.

> A **Tucker Decomposition** of a tensor $\Gamma$ framed by *axes* (known also as *modes*, *legs*, or *indices*) $\mathbb{K}^{d_1},\ldots, \mathbb{K}^{d_{\ell}}$, is a subset $A\subset\{1,\ldots,\ell\}$ and a decomposition
> $$\forall a\in A,\qquad \mathbb{K}^{d_a}=E_a\oplus R_a$$
> such that the contraction of $\Gamma$ on $R_a$ is 0.  

The decompositions can be given by partitioned bases of $E_a$ and $R_a$, or equivalently by linear projections $e_a:\mathbb{K}^{d_a}\to \mathbb{K}^{e_a}$ with kernel $R_a$.  In many situations the purpose of a Tucker decomposition is to restrict the tensor to the $E_a$-spaces, which we call the "truncated" form. 

### 1.1 Loading Dleto

This notebook uses the Python package in this checkout. Run the setup cell below once, then run the examples with a Python 3.10+ kernel.


### Local Python setup

The editable install includes NumPy, SciPy, and the optional 3D plotting dependency.


In [ ]:
# Colab opens the notebook without its repository files; fetch them once.
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

candidates = [Path.cwd(), *Path.cwd().parents]
candidates += [base / "OpenDleto" for base in candidates]
if os.environ.get("OPENDLETO_ROOT"):
    candidates.insert(0, Path(os.environ["OPENDLETO_ROOT"]).expanduser())
root = next((p for p in candidates if (p / "python/src/opendleto").is_dir()
             and (p / "labs/FPED_1720.csv").is_file()), None)
cloned = root is None
if cloned:
    root = Path.cwd() / "OpenDleto"
    subprocess.check_call(["git", "clone", "--depth", "1", "--branch", "main",
                           "https://github.com/thetensor-space/OpenDleto.git", str(root)])
if cloned or any(importlib.util.find_spec(name) is None
                 for name in ("numpy", "scipy", "plotly")):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-e",
                           str(root / "python") + "[lab]"])
    importlib.invalidate_caches()
sys.path.insert(0, str(root / "python/src"))

Once you have installed the Python package you can load NumPy and Dleto. `plotly` supplies interactive 3D plots. The helper `apply_transforms(Γ, Xs)` makes reconstruction checks explicit.


In [ ]:
import gc
import time
import numpy as np
from opendleto import (apply_transforms, chisel_tensor, compare, direct_sum,
                       nondeg, plot_tensor, randomize_tensor)
rng = np.random.default_rng(42)

Select your viewer properties, `False` for laptop & desktop viewing and `True` for vertical screens like tablets/phones:

In [ ]:
vertical = False; pic_size = (900, 400)
# vertical = True; pic_size = (400, 900)

### 1.2 Creating a Tensor Experiment

We create two tensors, one randomized for control, and one identical in size but with 2 rows, 2 columns, and 2 slices set to approximately zero.  We then randomize the experiment tensor by applying a change in coordinates. We will demostrate how to use chiseling to detect and recover the hidden zero rows / columns / slices.

> **Notation.** Python 3 accepts Unicode variable names; many notebook editors also offer $\LaTeX$ styled tab completion.  For example, to insert the Unicode character for $\Gamma$ use `\Gamma` in the code area followed by `tab` (or cut-and-paste a character you see somewhere else). If you prefer, you can replace these characters with strings of characters of your choosing. `opendleto` calculations make substantial use of tensors, matrices, and lists of matrices, so we recommend using a convention that clearly delineates roles.  
>
> We will use:
> * Greek capitals such as `Γ` (`\Gamma`), `Δ` (`\Delta`), `Σ` (`\Sigma`), `Ξ` (`\Xi`), `Υ` (`\Upsilon`) for tensors;
> * English capitals such as `X`, `Y`, `Z` for matrices;
> * Greek lower-case letters for real numbers;
> * English lower-case letters for integers; and
> * variables that suggest plurality, such as  `Γs` and `Xs`, for lists.

The Python Dleto implementation works directly with NumPy arrays. A `TensorElement` wrapper is available, but the arrays below need no conversion.

In [ ]:
ds = (10, 8, 5)
Γ = rng.normal(size=ds)

es = (5, 5, 5)
Δ = direct_sum(rng.normal(size=es), np.zeros(tuple(d - e for d, e in zip(ds, es))))
assert Γ.shape == Δ.shape == ds

Visualization tools are very useful for larger tensors:

In [ ]:
compare(Γ, Δ, left_title="Control Γ", right_title="Experiment Δ", vertical=vertical).show()

The control tensor is unlikely to have large regions of zeros, while the experiment tensor is constructed so that it does. Both tensors have shape `(7, 6, 5)`; the sparse experiment has only a `(5, 5, 5)` block of nonzero entries. Plotly axes therefore span the full zero-based tensor coordinates, including the zero rows and columns. You will observe changes when you adjust the parameters.

### 1.3 Tensor Randomization
Dleto detects features such as redundancy without knowing about them in advance. We provide a demonstration. First, we scramble the experiment tensor to obscure its features. The scrambling process applies random basis changes to each of the original coordinate axes.

With the NumPy arrays we state reconstruction checks explicitly as
$$
\texttt{apply\_transforms}(\Gamma, X\_s) \approx \Gamma\_{\mathrm{rand}}.
$$

Dleto applies one coordinate matrix per axis, so this notebook can stay focused on the algebra and interpretation.

In [ ]:
Γ_rand, X1s = randomize_tensor(Γ, rng=rng)
assert np.allclose(apply_transforms(Γ, X1s), Γ_rand)

Δ_rand, Y1s = randomize_tensor(Δ, rng=rng)
assert np.allclose(apply_transforms(Δ, Y1s), Δ_rand)


The control and the experiment tensor now look similarly unstructured:

In [ ]:
tol = 1e-5 # a small tolerance for numerical comparisons
compare(Γ_rand, Δ_rand, tol, left_title="Control Γ", right_title="Experiment Δ_rand", vertical=vertical).show()

### 1.4 Chiseling for Tucker Decompositions

We now use Dleto to find a Tucker decomposition.  The key command is `stratify`.  While this function can be used with multiple parameters, it has default settings that detect structure when it is present and often perform well as a first approximation.  

In [ ]:
Γ_result = chisel_tensor(Γ_rand, tol=tol)
Γ_strat, X2s = Γ_result.tensor, Γ_result.transforms
assert np.allclose(apply_transforms(Γ_rand, X2s), Γ_strat)
Δ_result = chisel_tensor(Δ_rand, tol=tol)
Δ_strat, Y2s = Δ_result.tensor, Δ_result.transforms
assert np.allclose(apply_transforms(Δ_rand, Y2s), Δ_strat)
print("Nontrivial derivations (control, experiment):", Γ_result.derivation_count, Δ_result.derivation_count)

The output is a tensor-transform pair. You may observe some extra information printed that reports the number of nontrivial "derivations" detected during stratification. Random dense tensors typically admit only scalar derivations, which the Python count omits. When a Tucker decomposition exists in a tensor, on the other hand, you will typically see more than 0 derivations. Indeed you can think of a derivation number greater than 0 as a proxy for `Dleto` having detected something of interest in your tensor.

**Round 2?** If you modified the parameters you may observe that stratification no longer distinguishes the experiment tensor. You now need to consider how much tolerance to pass along to the stratification by adding the optional parameter `tol=tol` (the tolerance we set above).  Replace the original commands by the commented commands to include this tolerance adjustment.  You will know that your tolerance is adequate if the number derivations for the experiment is more than 0.

---

Let us look at the results:

In [ ]:
compare(Γ_strat, Δ_strat, tol, left_title="Stratified Control Γ", right_title="Stratified Experiment Δ", vertical=vertical).show()

If the noise-to-dimension ratio is reasonable, you should observe that the experiment tensor now clusters nearly all its values in one region. Although the location of that region may shift from experiment to experiment, it should have similar volume to the initial nonzero region in the experiment tensor. 

This is a good place to return to the start of this [section](#12-creating-a-tensor-experiment) and begin altering the parameters to see how the results change.  
 * First you could increase the dimension. <br> **Caution:** the complexity of a tensor is proportional to the volume.  Changing (5,5,5) to (25,25,25) increases the input by a factor of 125, not 5! You should therefore proceed with caution until you acclimate to the impact your changes have on performance and memory.
 * Secondly, you could increase the background noise and set the tolerence to match. For example, `tol=2*db` is an appropriate range.

### 1.5 Optimal Tucker Decompositions

As noted earlier, Dleto has a function `nondeg` to perform Tucker computations more efficiently. The function accepts two modes `"full"` to regroup the nonzero values inside the original space, and `"trunc"` to truncate the detected zero rows, columns, slices.  It is usually advisable to pass to a nondegenerate tensor using `nondeg` at the start of any computation because it has a relatively low cost and the dimension of the resulting tensor can be (sometimes quite substantially) smaller. This can reduce the cost of all subsequent computations.

In [ ]:
Γ_reduction = nondeg(Γ_rand, mode="full")
Γ_nondeg, X3s = Γ_reduction.core, tuple(b.conj().T for b in Γ_reduction.bases)
assert np.allclose(apply_transforms(Γ_rand, X3s), Γ_nondeg)
Δ_reduction = nondeg(Δ_rand, mode="full")
Δ_nondeg, Y3s = Δ_reduction.core, tuple(b.conj().T for b in Δ_reduction.bases)
assert np.allclose(apply_transforms(Δ_rand, Y3s), Δ_nondeg)
compare(Γ_nondeg, Δ_nondeg, tol, left_title="Tucker Control Γ", right_title="Tucker Experiment Δ", vertical=vertical).show()

In [ ]:
Δ_full = nondeg(Δ_rand, mode="full")
Δ_nondeg = Δ_full.core
Δ_trunc = nondeg(Δ_rand, mode="trunc")
Δ_nondeg_trunc, Y4s = Δ_trunc.core, tuple(b.conj().T for b in Δ_trunc.bases)
assert np.allclose(apply_transforms(Δ_rand, Y4s), Δ_nondeg_trunc)
print("Full and truncated shapes:", Δ_nondeg.shape, Δ_nondeg_trunc.shape)
compare(Δ_nondeg, Δ_nondeg_trunc, tol, left_title="Tucker Experiment Δ full", right_title="Tucker Experiment Δ truncated", vertical=vertical).show()

Tucker decompositions are closures: repeated applications achieve no further gains (though it may reorder and rescale the one its has already carried out):  

In [ ]:
Δ_nondeg_trunc = nondeg(Δ_rand, mode="trunc").core
Δ_nondeg2_trunc = nondeg(Δ_nondeg_trunc, mode="trunc").core
assert Δ_nondeg_trunc.shape == Δ_nondeg2_trunc.shape

If you did not already change parameters above, we invite you to return to the first cell and do so now. We can continue to other experiments when you are ready!

-----

### (Optional) Carry out the trash

While it is helpful to visualize the effects of Dleto chiseling, these 3D plots (especially of large dense randomized data) can use up much the available memory of a notebook environment.  If you have limited resources to run the notebook or experience lag times then you may benefite from running the following Python garbage collection routine between sections. If you are very tight on memory budgets you may wish to clear output data of the 3D plots above.

In [ ]:
for name in ("Γ", "Δ", "Γ_rand", "Δ_rand", "Γ_strat", "Δ_strat", "X1s", "Y1s", "X2s", "Y2s"):
    globals().pop(name, None)
gc.collect()

# 2. Block Diagonalization

You can think of performing a Tucker decomposition as finding a single block on the diagonal of a tensor. It is of course natural to ask whether one can find several distinct blocks (assuming they exist). In similar fashion to the first experiment, we now create such a tensor and hide the blocks to see if Dleto chiseling can recover them.

In [ ]:
as_ = [3, 2, 7]
bs = [4, 5, 2]
cs = [5, 3, 3]
ds = tuple(a + b + c for a, b, c in zip(as_, bs, cs))

Γ = rng.normal(size=ds) # a control tensor
Δ = direct_sum(rng.normal(size=as_), rng.normal(size=bs), rng.normal(size=cs))

# Add noise to experiment tensor
# db = 0.001; tol = 10*db
# Δ += db*rng.normal(size=Δ.shape)
compare(Γ, Δ, left_title="Control Γ", right_title="Experiment Δ", vertical=vertical).show()

We again hide the structure by randomizing the reference frame of our tensor:

In [ ]:
Γ_rand, X1s = randomize_tensor(Γ, rng=rng)
Δ_rand, Y1s = randomize_tensor(Δ, rng=rng)
assert np.allclose(apply_transforms(Δ, Y1s), Δ_rand)
compare(Γ_rand, Δ_rand, left_title="Scrambled Control Γ", right_title="Scrambled Experiment Δ", vertical=vertical).show()

This at least provides visual assurance that we have hidden the block structure in our experiment tensor. Let us now try to use chiseling to recover it: 

In [ ]:
# If you added noise a couple blocks up, adjust this tolerance to match it.
# tol = 1e-3
Γ_result = chisel_tensor(Γ_rand, tol=tol)
Γ_strat, X2s = Γ_result.tensor, Γ_result.transforms
assert np.allclose(apply_transforms(Γ_rand, X2s), Γ_strat)
Δ_result = chisel_tensor(Δ_rand, tol=tol)
Δ_strat, Y2s = Δ_result.tensor, Δ_result.transforms
assert np.allclose(apply_transforms(Δ_rand, Y2s), Δ_strat)
print("Nontrivial derivations (control, experiment):", Γ_result.derivation_count, Δ_result.derivation_count)

The volume of the dots are proportional to the scalars they represent. In particular, tiny values are omitted entirely from the plots. Inspecting the actual data provides more detail. Up to a small tolerance we see that we have recovered the degeneracy embedded in the experimental tensor.

In [ ]:
compare(Γ_strat, Δ_strat, tol, left_title="Stratified Control Γ", right_title="Stratified Experiment Δ", vertical=vertical).show()

Once more we have recovered some structure.  The order of our blocks has changed (and it may be helpful to interact with 3D plot to understand the structure).


We again invite you to revisit the experiment and add noise, add more blocks, and change dimensions.

### Carry out the trash
If you are ready for the next experiment you might get better viewing results if you release some more trash.

In [ ]:
for name in ("Γ", "Δ", "Γ_rand", "Δ_rand", "Γ_strat", "Δ_strat", "X1s", "Y1s", "X2s", "Y2s"):
    globals().pop(name, None)
gc.collect()

## 3 Stratification

In linear algebra there are many uses for diagonals, block diagonals, and triangular forms of matrices. Moving from matrices to tensors with more than two axes, we encounter something new and unrelated to matrices: curves and surfaces.

In keeping with previous demonstrations, we carry out a controlled experiment.

In [ ]:
ds = (30, 30, 30)
Γ = rng.normal(size=ds) # a control tensor

db = 0.0
tol = 1e-8
surface_rng = np.random.default_rng(42)  # stable if earlier cells are rerun

# An experiment tensor with a hidden surface.
# Choose f(x,y,z) to define the surface.
def f(x, y, z):
    return x**2 + y**2 + z**2

# Populate Δ near the surface defined by f using one-based coordinates.
i, j, k = np.ogrid[1:ds[0]+1, 1:ds[1]+1, 1:ds[2]+1]
Δ = np.where(np.abs(f(i, j, k) - 30**2) < 1 + 4*surface_rng.normal(size=ds),
             surface_rng.normal(size=ds), 0.0)
assert Γ.dtype == Δ.dtype == np.float64  # SciPy solvers use double precision here
compare(Γ, Δ, left_title="Control Γ", right_title="Experiment Δ", vertical=vertical).show()

Dleto supplies a range of convenient functions to generate tensors of this sort. It also provides configurable noise and distance functions.

In [ ]:
Γ_rand, X1s = randomize_tensor(Γ, rng=rng)
assert np.allclose(apply_transforms(Γ, X1s), Γ_rand)
Δ_rand, Y1s = randomize_tensor(Δ, rng=rng)
assert np.allclose(apply_transforms(Δ, Y1s), Δ_rand)

The randomized versions are by now visually familiar:

In [ ]:
compare(Γ_rand, Δ_rand, tol, left_title="Scrambled Control Γ", right_title="Scrambled Experiment Δ", vertical=vertical).show()

In [ ]:
start = time.perf_counter()
Γ_result = chisel_tensor(Γ_rand, tol=tol)
Γ_strat, X2s = Γ_result.tensor, Γ_result.transforms
print(f"Control stratification: {time.perf_counter()-start:.2f}s; exact={Γ_result.exact}; nontrivial derivations={Γ_result.derivation_count}")
assert np.allclose(apply_transforms(Γ_rand, X2s), Γ_strat)

Since we are trying to stratify larger surfaces, it makes sense (as we suggested earlier) to first remove any degeneracies:

In [ ]:
Δ_reduction = nondeg(Δ_rand, mode="trunc")
Δ_nondeg, Y2s = Δ_reduction.core, tuple(b.conj().T for b in Δ_reduction.bases)
assert np.allclose(apply_transforms(Δ_rand, Y2s), Δ_nondeg)
start = time.perf_counter()
Δ_result = chisel_tensor(Δ_nondeg, tol=tol)
Δ_strat, Z1s = Δ_result.tensor, Δ_result.transforms
print(f"Verified nontrivial derivations: {Δ_result.derivation_count}")
print(Δ_result.solver_message)
if Δ_result.derivation_count == 0:
    raise RuntimeError(f"No verified nontrivial derivations for core shape {Δ_nondeg.shape}; "
                       "restart the kernel and rerun the notebook from the setup cell")
print(f"Experiment stratification: {time.perf_counter()-start:.2f}s; exact={Δ_result.exact}; nontrivial derivations={Δ_result.derivation_count}")
print(f"Law error={Δ_result.relative_law_error:.2e}; forbidden energy={Δ_result.forbidden_energy_fraction}; "
      f"predicted support={Δ_result.support_count}; original nonzeros={np.count_nonzero(Δ)}; "
      f"frame condition={Δ_result.frame_condition}")
assert np.allclose(apply_transforms(Δ_nondeg, Z1s), Δ_strat)

In [ ]:
compare(Γ_strat, Δ_strat, tol, left_title="Stratified Control Γ", right_title="Stratified Experiment Δ", vertical=vertical).show()